In [ ]:
import copy
import glob
import json
import os
import random
import re
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, List
import itertools
import scipy as sp
from collections import defaultdict

import h5py
import imageio.v3 as iio
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cycler
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Circle, Polygon
from numba import njit
from scipy.interpolate import CubicSpline
from scipy.ndimage import uniform_filter1d
import scipy as sp

# plt.style.use("ggplot")
# plt.rcParams["figure.figsize"] = (12, 10)
spec_home = "/home/himanshu/spec/my_spec"
matplotlib.matplotlib_fname()


# Idea
Have a single python class that can read all the data from a SpEC simulation into pandas/duckdb for plotting.


- In general having all the levs of a run is very good for comparitive plotting.
- Or just have the capacity to read and join that comes from a regex of Levs, the another class deals with single runs multiple levs stuff
- Have a base single segment reader, then another function can join any required segments and finally that one that can deal with multiple different runs

# Functions

## Readers

In [ ]:
def dat_files_to_df(file_name):
    """
    We first get the column names from the file. Once we have the column names, we can treat them as comments and read rest of file as a regular CSV.
    """
    cols_names = []
    # Read column names
    with open(file_name, "r") as f:
        lines = f.readlines()
        for line in lines:
            if "#" not in line:
                # From now onwards it will be all data
                break
            elif "=" in line:
                if ("[" not in line) and ("]" not in line):
                    # columns names are always like [1] = "column_name"
                    continue
                cols_names.append(line.split("=")[-1][1:-1].strip())
            else:
                continue

    return pd.read_csv(file_name, sep="\s+", comment="#", names=cols_names)


def hist_files_to_df(file_path):
    # Function to parse a single line and return a dictionary of values
    def parse_line(line):
        data = {}
        # Find all variable=value pairs
        pairs = re.findall(r"([^;=\s]+)=\s*([^;]+)", line)
        for var, val in pairs:
            # Hist-GrDomain.txt should be parsed a little differently
            if "ResizeTheseSubdomains" in var:
                items = val.split("),")
                items[-1] = items[-1][:-1]
                for item in items:
                    name, _, vals = item.split("(")
                    r, l, m = vals[:-1].split(",")
                    data[f"{name}_R"] = int(r)
                    data[f"{name}_L"] = int(l)
                    data[f"{name}_M"] = int(m)
            else:
                data[var] = float(val) if re.match(r"^[\d.e+-]+$", val) else val
        return data

    with open(file_path, "r") as file:
        # Parse the lines
        data = []
        for line in file.readlines():
            data.append(parse_line(line.strip()))

        # Create a DataFrame
        df = pd.DataFrame(data)

    return df


In [ ]:
Lev_path = Path("/resnick/groups/sxs/hchaudha/spec_runs/high_accuracy_L35_master/Ev/Lev5_AB/Run")

# Find all the .dat, .hist, and .h5 files in the directory
dat_files = list(Lev_path.glob("**/*.dat"))
hist_files = list(Lev_path.glob("**/Hist-*.txt"))
h5_files = list(Lev_path.glob("**/*.h5"))


#### Test on all the dat files

In [ ]:
# Make sure we can read all the dat files
# Also print size of the file, time to parse, and size of the df
for file in dat_files:
    try:
        start_time = time.time()
        df = dat_files_to_df(file)
        elapsed_time = time.time() - start_time
        file_size = file.stat().st_size / (1024 * 1024)  # Size in MB
        # df size in memory
        df_memory_size = df.memory_usage(deep=True).sum() / (1024 * 1024)  # Size in MB
        # Print such that all the time and size are aligned
        print(f"{str(file.relative_to(Lev_path)):<55}: {elapsed_time:.4f}s, {file_size:.3f}MB, {df_memory_size:.3f}MB, {file_size/elapsed_time:.3f}MB/s")
    except Exception as e:
        print(f"Error reading {file}: {e}")

#### test on all the hist files

In [ ]:
# Make sure we can read all the hist files
# Also print size of the file, time to parse, and size of the df
for file in hist_files:
    try:
        start_time = time.time()
        df = hist_files_to_df(file)
        elapsed_time = time.time() - start_time
        file_size = file.stat().st_size / (1024 * 1024)  # Size in MB
        # df size in memory
        df_memory_size = df.memory_usage(deep=True).sum() / (1024 * 1024)  # Size in MB
        # Print such that all the time and size are aligned
        print(f"{str(file.relative_to(Lev_path)):<55}: {elapsed_time:.4f}s, {file_size:.3f}MB, {df_memory_size:.3f}MB, {file_size/elapsed_time:.3f}MB/s")
    except Exception as e:
        print(f"Error reading {file}: {e}")

#### h5 files

In [ ]:
h5_files

In [ ]:
file_path = h5_files[0]
with h5py.File(file_path, "r") as f:
    print(file_path.stem)
    print(f['Step40839.dir'].keys())